# 04 — CNN 2D sobre log-mel espectrogramas

Usa as entradas de `02_preprocessing.ipynb` (`data/processed/mel.npy`):
- **Entrada**: log-mel 40 × 401 por janela de 4 s (n_fft 256 = 64 ms, hop 10 ms, 20–1000 Hz), em dB,
  cortada em tempo real de `mel.npy` (`CNNWindows`) e normalizada **por banda mel com estatísticas só do treino**.
  Não há normalização por janela: o nível absoluto ajuda a reconhecer gravações ruidosas (*Unknown*, 01_eda §8).
- **Aumento de dados**: deslocamento temporal aleatório de até ±meia janela em cada época (`random_crop`) + SpecAugment
  (uma máscara de frequência ≤ 8 bandas e uma de tempo ≤ 0.4 s por batch).
- **Modelo**: 4 blocos Conv 3×3 com stride 2 (16 → 32 → 64 → 128 mapas) + BatchNorm + ReLU, *global average pooling*,
  cabeça 128 → 64 → 3 com dropout 0.3.
- **Treino**: cross-entropy com pesos de classe, Adam, early stopping na loss de validação. Rótulo por gravação (`rec_label`).
- **Uma previsão por paciente**: igual à MLP (`src/evaluate.py`).

O treino pode correr aqui (`TRAIN = True`) ou noutro servidor com `python -m src.models.cnn` (GPU se existir), que grava em `results/cnn/`.

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import CLASSES
from src.datasets import load_processed
from src.evaluate import RESULTS_DIR, result_dir, patient_scores, cv_report, tune_thresholds, predict_rule, confusion_df, save_oof, load_oof, summary
from src.metrics import report
pd.set_option('display.width', 140)
d = load_processed()

## Uma entrada da CNN

In [ ]:
from src.datasets import fold_masks, mel_band_stats, CNNWindows
tr, va, te = fold_masks(d.windows, val_fold=0)
ds = CNNWindows(d, va, mel_band_stats(d, tr))
fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for c, ax in zip(CLASSES, axes):
    i = int(np.flatnonzero(ds.rows.rec_label == c)[0]); x, _ = ds[i]
    ax.imshow(x[0], origin='lower', aspect='auto', cmap='magma', extent=[0, 4, 0, 40])
    ax.set(title=f'{c} — {ds.rows.recording[i]}', xlabel='s', ylabel='banda mel')
plt.tight_layout(); print('entrada:', tuple(x.shape))

## Treino (validação cruzada, um modelo por fold)

In [ ]:
from src.models.cnn import DEFAULT_CNN_CFG, cross_validate_cnn, save_runs, load_runs, predict_ensemble_cnn
NAME, TRAIN = 'cnn', False
cfg = dict(DEFAULT_CNN_CFG)

if TRAIN:
    dev, oof, runs = cross_validate_cnn(d, cfg)
    t_, _, r_ = cv_report(patient_scores(dev, oof))
    out = save_oof(NAME, dev.index, oof, summary(t_, r_) | {'cfg': cfg, 'thresholds': tune_thresholds(patient_scores(dev, oof))[0]})
    save_runs(runs, cfg, out)
    hist = {r['fold']: r['history'] for r in runs}
else:
    idx, oof = load_oof(NAME)
    dev = d.windows.loc[idx]
    hist = {int(p.stem.split('fold')[1]): pd.read_csv(p) for p in sorted((result_dir(NAME)).glob('history_fold*.csv'))}
    print(json.loads((result_dir(NAME) / 'cv_summary.json').read_text())['cfg'])
print(len(dev), 'janelas de validação (out-of-fold)')

In [ ]:
fig, axes = plt.subplots(1, len(hist), figsize=(3.6 * len(hist), 3), sharey=True)
for (k, h), ax in zip(hist.items(), np.atleast_1d(axes)):
    ax.plot(h.epoch, h.train_loss, label='treino'); ax.plot(h.epoch, h.val_loss, label='validação')
    ax.axvline(h.val_loss.idxmin(), ls=':', c='k'); ax.set(title=f'fold {k}', xlabel='época')
np.atleast_1d(axes)[0].set_ylabel('CE ponderada'); np.atleast_1d(axes)[0].legend(); plt.tight_layout()

## Avaliação por paciente

In [ ]:
AGG = 'max'

pat = patient_scores(dev, oof, how=AGG)
table, pred, rep = cv_report(pat)
display(table.round(3))
print('WA (limiares afinados nos outros folds): %.3f ± %.3f | pooled %.3f' % (table['WA (tuned)'].mean(), table['WA (tuned)'].std(), rep['weighted_accuracy']))
print('recall:', {c: round(v, 3) for c, v in rep['recall'].items()})
confusion_df(rep)

### Agregação por paciente

In [ ]:
rows = []
for how in ['max', 'max_present', 'mean']:
    t, _, r = cv_report(patient_scores(dev, oof, how=how))
    rows.append({'agregação': how, 'WA média': t['WA (tuned)'].mean(), 'WA std': t['WA (tuned)'].std(), **{f'recall {c}': r['recall'][c] for c in CLASSES}})
pd.DataFrame(rows).set_index('agregação').round(3)

In [ ]:
t_final, wa_oof = tune_thresholds(pat)
print('t_present = %.3f, t_unknown = %.3f (WA OOF %.3f, otimista)' % (*t_final, wa_oof))

## Teste — correr só uma vez, no fim

In [ ]:
RUN_TEST = False
if RUN_TEST:
    runs = load_runs(result_dir(NAME))
    te = (d.windows.fold == -1).to_numpy()
    pat_te = patient_scores(d.windows[te], predict_ensemble_cnn(runs, d, te, cfg), how=AGG)
    r = report(pat_te.murmur, predict_rule(pat_te, *t_final))
    print('TESTE WA = %.3f' % r['weighted_accuracy'], {c: round(v, 3) for c, v in r['recall'].items()})
    display(confusion_df(r))